In [0]:
display(spark.sql("""
SELECT parameter_or_file, status, rows_inserted, left(error_message, 200) AS error, start_time
FROM workspace.rag.pipeline_execution_logs
WHERE layer = 'Source-to-Raw'
ORDER BY start_time DESC
"""))

display(spark.sql("SELECT _source_repo, 'commits' AS t, COUNT(*) AS n FROM workspace.rag.bronze_commits GROUP BY 1 UNION ALL SELECT _source_repo, 'issues', COUNT(*) FROM workspace.rag.bronze_issues GROUP BY 1 ORDER BY 1, 2"""))

display(spark.sql("SELECT entity, COUNT(*) AS n FROM workspace.rag.silver_quarantine GROUP BY 1"))
display(spark.sql("SELECT endpoint, COUNT(*) AS n FROM workspace.rag.bronze_quarantine GROUP BY 1"))

In [0]:
def folder_size(path):
    total = 0
    for f in dbutils.fs.ls(path):
        total += folder_size(f.path) if f.path.endswith("/") else f.size
    return total

base = "/Volumes/workspace/rag/raw/landing"
for sub in ["full/commits", "full/issues", "incremental"]:
    try:
        print(f"RAW files {sub}: {folder_size(f'{base}/{sub}')/1024/1024:.1f} MB (includes duplicate test pulls)")
    except Exception as e:
        print(sub, "-", str(e)[:80])

display(spark.sql("""
SELECT 'commits' AS tbl, COUNT(*) AS unique_rows, ROUND(SUM(len)/1024/1024, 1) AS unique_raw_json_mb FROM (
  SELECT _source_repo, sha, MAX(length(_raw_json)) AS len FROM workspace.rag.bronze_commits GROUP BY 1, 2)
UNION ALL
SELECT 'issues', COUNT(*), ROUND(SUM(len)/1024/1024, 1) FROM (
  SELECT _source_repo, id, MAX(length(_raw_json)) AS len FROM workspace.rag.bronze_issues GROUP BY 1, 2)
"""))

display(spark.sql("SELECT repo_name, COUNT(*) AS silver_commits FROM workspace.rag.silver_commits GROUP BY 1 ORDER BY 1"))
display(spark.sql("SELECT repo_name, COUNT(*) AS silver_issues FROM workspace.rag.silver_issues GROUP BY 1 ORDER BY 1"))

In [0]:
display(spark.sql("""
SELECT parameter_or_file, left(error_message, 300) AS error, start_time
FROM workspace.rag.pipeline_execution_logs
WHERE layer = 'Source-to-Raw' AND status = 'Failure'
ORDER BY start_time DESC LIMIT 10
"""))